 ## 1. Чтение и импорты

In [1]:
import pandas as pd
import gc

In [2]:
df = pd.read_csv("../ex04/fines.csv")
df

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1,2100.0,Ford,Focus,1984
3,X582HE161RUS,2,2000.0,Ford,Focus,2015
4,92918M178RUS,1,5700.0,Ford,Focus,2014
...,...,...,...,...,...,...
925,A111AA77RUS,0,1800.0,Toyota,Camry,2007
926,B222BB99RUS,1,2500.0,BMW,X5,2007
927,C333CC177RUS,0,900.0,Kia,Rio,2007
928,D444DD197RUS,1,3200.0,Audi,A4,2007


## 2. Считаем новый столбец разными способами 

- Обычный луп

In [3]:
%%timeit
loop_res = []
for i in range(len(df)):
    loop_res.append(df.iloc[i]["Fines"] / df.iloc[i]["Refund"] * df.iloc[i]["Year"])
df["calc_loop"] = loop_res

<magic-timeit>:3: RuntimeWarning: divide by zero encountered in scalar divide
<magic-timeit>:3: RuntimeWarning: divide by zero encountered in scalar divide


25.5 ms ± 393 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


- itterrows()

In [4]:
%%timeit

iter_res = []
for _, row in df.iterrows():
    try:
        iter_res.append(row["Fines"] / row["Refund"] * row["Year"])
    except:
        iter_res.append(None)


df["calc_iterrows"] = iter_res


9.33 ms ± 211 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- apply + lambda

In [5]:
%%timeit
df["calc_apply"] = df.apply(lambda row: row["Fines"] / row["Refund"] * row["Year"] if row["Refund"]!= 0 else None, axis=1)
df

3.11 ms ± 24 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- Series

In [6]:
%%timeit
df["calc_series"] = df["Fines"] / df["Refund"] * df["Year"]
df

50.6 μs ± 188 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


- Series with .values()

In [7]:
%%timeit
df["calc_values"] = df["Fines"].values / df["Refund"].values * df["Year"].values


<magic-timeit>:1: RuntimeWarning: divide by zero encountered in divide
<magic-timeit>:1: RuntimeWarning: divide by zero encountered in divide


25.4 μs ± 120 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [8]:
df

,CarNumber,Refund,Fines,Make,Model,Year,calc_loop,calc_iterrows,calc_apply,calc_series,calc_values
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989,3182400.0,3182400.0,3182400.0,3182400.0,3182400.0
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995,12967500.0,12967500.0,12967500.0,12967500.0,12967500.0
2,7184TT36RUS,1,2100.0,Ford,Focus,1984,4166400.0,4166400.0,4166400.0,4166400.0,4166400.0
3,X582HE161RUS,2,2000.0,Ford,Focus,2015,2015000.0,2015000.0,2015000.0,2015000.0,2015000.0
4,92918M178RUS,1,5700.0,Ford,Focus,2014,11479800.0,11479800.0,11479800.0,11479800.0,11479800.0
...,...,...,...,...,...,...,...,...,...,...,...
925,A111AA77RUS,0,1800.0,Toyota,Camry,2007,inf,NaN,NaN,inf,inf
926,B222BB99RUS,1,2500.0,BMW,X5,2007,5017500.0,5017500.0,5017500.0,5017500.0,5017500.0
927,C333CC177RUS,0,900.0,Kia,Rio,2007,inf,NaN,NaN,inf,inf
928,D444DD197RUS,1,3200.0,Audi,A4,2007,6422400.0,6422400.0,6422400.0,6422400.0,6422400.0


## 3. Получение данных по индексам

In [9]:
%%timeit
df[df["CarNumber"] == "O136HO197RUS"]

83.2 μs ± 196 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [10]:

df = df.set_index("CarNumber")
df

,Refund,Fines,Make,Model,Year,calc_loop,calc_iterrows,calc_apply,calc_series,calc_values
CarNumber,,,,,,,,,,
Y163O8161RUS,2,3200.0,Ford,Focus,1989,3182400.0,3182400.0,3182400.0,3182400.0,3182400.0
E432XX77RUS,1,6500.0,Toyota,Camry,1995,12967500.0,12967500.0,12967500.0,12967500.0,12967500.0
7184TT36RUS,1,2100.0,Ford,Focus,1984,4166400.0,4166400.0,4166400.0,4166400.0,4166400.0
X582HE161RUS,2,2000.0,Ford,Focus,2015,2015000.0,2015000.0,2015000.0,2015000.0,2015000.0
92918M178RUS,1,5700.0,Ford,Focus,2014,11479800.0,11479800.0,11479800.0,11479800.0,11479800.0
...,...,...,...,...,...,...,...,...,...,...
A111AA77RUS,0,1800.0,Toyota,Camry,2007,inf,NaN,NaN,inf,inf
B222BB99RUS,1,2500.0,BMW,X5,2007,5017500.0,5017500.0,5017500.0,5017500.0,5017500.0
C333CC177RUS,0,900.0,Kia,Rio,2007,inf,NaN,NaN,inf,inf


In [11]:
%%timeit
df.loc["O136HO197RUS"]


34.7 μs ± 595 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 4. Downcasting

- смотрим типы данных

In [12]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int64  
 1   Fines          930 non-null    float64
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int64  
 5   calc_loop      930 non-null    float64
 6   calc_iterrows  927 non-null    float64
 7   calc_apply     927 non-null    float64
 8   calc_series    930 non-null    float64
 9   calc_values    930 non-null    float64
dtypes: float64(6), int64(2), object(2)
memory usage: 243.3 KB


- создаем копию датафрейма

In [13]:
optimized_df = df.copy()

- преобразовываем тип данных ф64 к ф32

In [14]:
float_cols = optimized_df.select_dtypes(include=["float64"]).columns
optimized_df[float_cols] = optimized_df[float_cols].astype("float32")


- преобразовываем тип данных интов к меньшему


In [15]:
optimized_df["Refund"] = pd.to_numeric(optimized_df["Refund"], downcast="integer")
optimized_df["Year"] = pd.to_numeric(optimized_df["Year"], downcast="integer")


In [16]:
optimized_df.info(memory_usage="deep")


<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int8   
 1   Fines          930 non-null    float32
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int16  
 5   calc_loop      930 non-null    float32
 6   calc_iterrows  927 non-null    float32
 7   calc_apply     927 non-null    float32
 8   calc_series    930 non-null    float32
 9   calc_values    930 non-null    float32
dtypes: float32(6), int16(1), int8(1), object(2)
memory usage: 209.7 KB


## 5. Теперь меняем тип у категориальных фич

In [17]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int8   
 1   Fines          930 non-null    float32
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int16  
 5   calc_loop      930 non-null    float32
 6   calc_iterrows  927 non-null    float32
 7   calc_apply     927 non-null    float32
 8   calc_series    930 non-null    float32
 9   calc_values    930 non-null    float32
dtypes: float32(6), int16(1), int8(1), object(2)
memory usage: 209.7 KB


In [18]:
optimized_df["Make"] = optimized_df["Make"].astype("category")
optimized_df["Model"] = optimized_df["Model"].astype("category")

In [19]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Refund         930 non-null    int8    
 1   Fines          930 non-null    float32 
 2   Make           930 non-null    category
 3   Model          919 non-null    category
 4   Year           930 non-null    int16   
 5   calc_loop      930 non-null    float32 
 6   calc_iterrows  927 non-null    float32 
 7   calc_apply     927 non-null    float32 
 8   calc_series    930 non-null    float32 
 9   calc_values    930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 115.5 KB


## 6. Memory clean

- смотрим сколько сейчас и какой датафрейм потребляет

In [20]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int64  
 1   Fines          930 non-null    float64
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int64  
 5   calc_loop      930 non-null    float64
 6   calc_iterrows  927 non-null    float64
 7   calc_apply     927 non-null    float64
 8   calc_series    930 non-null    float64
 9   calc_values    930 non-null    float64
dtypes: float64(6), int64(2), object(2)
memory usage: 243.3 KB


In [21]:
gc.collect()

7

In [22]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Refund         930 non-null    int64  
 1   Fines          930 non-null    float64
 2   Make           930 non-null    object 
 3   Model          919 non-null    object 
 4   Year           930 non-null    int64  
 5   calc_loop      930 non-null    float64
 6   calc_iterrows  927 non-null    float64
 7   calc_apply     927 non-null    float64
 8   calc_series    930 non-null    float64
 9   calc_values    930 non-null    float64
dtypes: float64(6), int64(2), object(2)
memory usage: 243.3 KB


In [23]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Refund         930 non-null    int8    
 1   Fines          930 non-null    float32 
 2   Make           930 non-null    category
 3   Model          919 non-null    category
 4   Year           930 non-null    int16   
 5   calc_loop      930 non-null    float32 
 6   calc_iterrows  927 non-null    float32 
 7   calc_apply     927 non-null    float32 
 8   calc_series    930 non-null    float32 
 9   calc_values    930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 115.5 KB


- После коллекта ничего не удалилось, так что применяем другую команду, причем обязательно с ^$ иначе она удалит всё где есть вхождения df

In [24]:
%reset_selective -f ^df$


In [25]:
optimized_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to E555EE750RUS
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Refund         930 non-null    int8    
 1   Fines          930 non-null    float32 
 2   Make           930 non-null    category
 3   Model          919 non-null    category
 4   Year           930 non-null    int16   
 5   calc_loop      930 non-null    float32 
 6   calc_iterrows  927 non-null    float32 
 7   calc_apply     927 non-null    float32 
 8   calc_series    930 non-null    float32 
 9   calc_values    930 non-null    float32 
dtypes: category(2), float32(6), int16(1), int8(1)
memory usage: 115.5 KB


In [26]:
df.info(memory_usage="deep")

NameError: name 'df' is not defined